# OSAI Project 1 — Colab v2.final Reproduction

Pascal-VOC 21-class semantic segmentation, ResNet-50 + DeepLabV3+ (OS=16).

**v2.final 솔루션** (improve 브랜치):
- **Stage 1** (160K iter, COCO+VOC): vanilla 학습 (= v2.0 equivalent)
- **Stage 2** (8K iter, VOC only): Copy-Paste augmentation + Boundary-weighted CE loss로 fine-tune

**ETA**: T4 ~14h (Stage 1 ~13h + Stage 2 ~1h), L4 ~10h

재현을 위해 다음 항목들을 수행해주셔야 합니다.

1. Drive에 폴더 생성: `MyDrive/osai/p1/submit/img/` (1000장 jpg)
2. WandB API key Colab Secret에 저장 (key 이름: `WANDB_API_KEY`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 설정 변수

In [ ]:
REPO_URL    = "https://github.com/geniemo/osai.git"
BRANCH      = "improve"
CONFIG_S1   = "src/config/colab_v2_final_s1.yaml"
CONFIG_S2   = "src/config/colab_v2_final_s2.yaml"
DRIVE       = "/content/drive/MyDrive/osai/p1"
CKPT_DIR    = f"{DRIVE}/checkpoints_v2_final"
CKPT_S2_BEST = f"{CKPT_DIR}/s2/best.pth"

import os
os.makedirs(f"{CKPT_DIR}/s1", exist_ok=True)
os.makedirs(f"{CKPT_DIR}/s2", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"Config S1: {CONFIG_S1}")
print(f"Config S2: {CONFIG_S2}")
print(f"Drive: {DRIVE}")
print(f"Ckpt dir: {CKPT_DIR}")
assert os.path.exists(f"{DRIVE}/submit/img"), f"submit/img not found at {DRIVE}/submit/img"
n_imgs = len([f for f in os.listdir(f"{DRIVE}/submit/img") if f.endswith('.jpg')])
print(f"submit/img: {n_imgs} jpg files")

## 2. 저장소 clone

In [ ]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

## 3. uv 설치 + 의존성 sync

In [ ]:
!pip install -q uv
!uv sync

## 4. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 5. WandB 로그인

In [ ]:
from google.colab import userdata
import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

!uv run wandb login

## 6. Test images Colab 로컬로 복사

In [ ]:
!mkdir -p submit/img
!cp -r {DRIVE}/submit/img/. submit/img/
!ls submit/img | wc -l
!ls submit/img | head -3

## 7. 데이터 다운로드 (VOC + COCO)

In [ ]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

## 8. COCO mask cache 사전 생성

95K instance mask를 PNG로 1회 생성 (~30-60분)

In [ ]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

## 9-1. Stage 1 학습 (vanilla, 160K iter)

COCO+VOC mixed, no Copy-Paste, no Boundary loss. ckpt가 Drive에 저장됨.
세션 끊기면 다시 시작 시 자동 resume.

**ETA**: T4 ~13h, L4 ~10h

In [ ]:
!uv run python -m src.train --config {CONFIG_S1} --stage 1

## 9-2. Stage 2 학습 (Copy-Paste + Boundary loss, 8K iter)

Stage 1 best ckpt에서 bootstrap. VOC train만 사용, Copy-Paste augmentation + Boundary-weighted CE loss로 fine-tune.

**ETA**: T4 ~1h, L4 ~30분

In [ ]:
!uv run python -m src.train --config {CONFIG_S2} --stage 2

## 10. Validation mIoU 측정

Stage 2 best ckpt로 val mIoU + 21 class별 IoU 출력.

In [ ]:
!uv run python -m src.eval --config {CONFIG_S2} --ckpt {CKPT_S2_BEST}

## 11. TTA Validation mIoU

Multi-scale [0.5, 0.75, 1.0, 1.25, 1.5] + hflip TTA. 실제 추론에 사용되는 성능.

In [ ]:
!uv run python -m src.eval_tta --config {CONFIG_S2} --ckpt {CKPT_S2_BEST}

## 12. 추론 — submit/img → submit/pred (TTA)

PDF 재현 컨벤션과 동일. ~5-10분 (T4).

In [ ]:
!mkdir -p submit/pred
!uv run python -m src.infer \
    --config {CONFIG_S2} \
    --ckpt {CKPT_S2_BEST} \
    --input submit/img \
    --output submit/pred

## 13. ONNX export (가중치 제거, 채점용 ONNX)

`model_structure.onnx` 생성 (~0.3MB, ≤10MB 채점 한도).

In [ ]:
!uv run python -m src.export_onnx \
    --config {CONFIG_S2} \
    --ckpt {CKPT_S2_BEST} \
    --out {DRIVE}/model_structure_v2_final.onnx

## 14. FLOPs 측정

In [ ]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure_v2_final.onnx

## 15. PNG zip 생성 (채점 사이트 제출용)

In [ ]:
!uv run python -m src.package_submission \
    --pred submit/pred \
    --out {DRIVE}/submission_pred_v2_final.zip

## 16. 결과물 확인

In [ ]:
!ls -la {DRIVE}/
!ls -la {CKPT_DIR}/s1/ {CKPT_DIR}/s2/